# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² open dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library. All dataset entities (record sets, fields, columns) are referenced by their `@id` as per the Croissant standard.

### Dataset Source
This dataset's Croissant schema is published at:
- [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Install mlcroissant if not present
!pip install -U mlcroissant

## 1. Data Loading
Load the Croissant schema, metadata, and discover dataset structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata
md = dataset.metadata  # This is a metadata object
print(f"Dataset: {md.name}\nDescription: {md.description}")


## 2. Data Overview
Preview available record sets (tables) and their fields and columns, referencing all by `@id` as per Croissant. This will help identify what data is available for extraction and how to reference it programmatically.

In [ ]:
# List all record sets and their @id
record_sets = list(dataset.record_sets.keys())
print("Record sets (@id):")
for rsid in record_sets:
    print(f"- {rsid}")
    rs = dataset.record_sets[rsid]
    print("  Fields:")
    for field in rs.fields:
        print(f"    • {field['@id']} (name: {field.get('name','')})")
    print("  Columns:")
    for col in rs.columns:
        print(f"    • {col['@id']} (name: {col.get('name','')})")
    print()

## 3. Data Extraction
For each record set, load its records into a pandas DataFrame using its `@id`, and present the available columns.

> **Note:** If the dataset is large, consider selecting a single record set or limiting the number of records for EDA.

In [ ]:
# Prepare extraction of every record set listed
# (Manual inspection may reveal the actual @ids. We'll use dynamic detection.)
dataframes = {}

for rsid in record_sets:
    print(f"\nExtracting records for record set: {rsid}")
    records = list(dataset.records(record_set=rsid))
    if len(records) == 0:
        print("  No records found.")
        continue
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print("  Fields (DataFrame columns):", list(df.columns))
    print(df.head(2).to_string())


## 4. Exploratory Data Analysis (EDA)
Let's select one available record set and numeric field (by `@id`) for further analysis.

We'll:
1. Filter records based on a threshold
2. Normalize a numeric field
3. Optionally, group by a categorical field

> **If you do not know the field @ids, run the previous cell to discover them and set the variable below accordingly.

In [ ]:
# Select target record set, numeric field, and grouping field by @id
# (Replace these if your dataset structure differs)
if dataframes:
    # Use the first record set with data
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Attempt to find numeric fields (fields with float/int dtype)
    numeric_candidates = df.select_dtypes(include=['number']).columns
    if len(numeric_candidates) == 0:
        print("No numeric fields found in record set.")
    else:
        numeric_field_id = numeric_candidates[0]  # Use first numeric field; change if wanting a different one
        print(f"Using numeric field: {numeric_field_id}")

        threshold = df[numeric_field_id].quantile(0.75)  # filter above Q3 for demonstration
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize selected numeric column
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to find a grouping field (categorical) not same as numeric
        group_candidates = [c for c in df.columns if df[c].dtype == 'object' and c != numeric_field_id]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
else:
    print("No dataframes were loaded; please check the record set extraction above.")

## 5. Visualization

Visualize the distribution of the selected numeric field using matplotlib or seaborn. If grouping was possible, also plot the grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals() and len(filtered_df) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=15, color='steelblue')
    plt.title(f'Distribution of {numeric_field_id} (filtered)')
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field_id and grouped_df were set, show a barplot
    if 'group_field_id' in locals() and 'grouped_df' in locals():
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'Average {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No filtered dataframe available for visualization. Check previous steps.')

## 6. Conclusion

This notebook provided an initial exploration of the FAIR² dataset using the mlcroissant library, with all table, field, and column references made via their Croissant `@id`. Typical EDA steps included filtering, normalizing, and visualizing a numeric variable from the selected record set.

- The Croissant standard enables exploration of rich metadata and structured datasets for reproducible science.
- For further statistical modeling or domain analysis, consult the data dictionary descriptions and field definitions accessed above.
- For more on the dataset context, see: Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya.
